In [20]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from api.config import padding_config
from api.utils.overlay import overlay_layers
from api.utils.preprocessing import clip_polygons

In [13]:
root = r"C:\Users\mrsam\Downloads\full_ba"

layers = [
    ('roads', 'roads'),
    ('buildings', 'buildings'),
    ('utilities', 'combined_utilities_ba'),
    ('other_green_areas', 'ba_green_areas_others'),
    ('pavements', 'ba_pavements'),
    ('trees_not_over_utilities', 'ba_trees_not_over_utilities'),
    ('trees_over_utilities', 'ba_trees_over_utilities'),
]

In [14]:
master_data = gpd.read_file(r"C:\Users\mrsam\Downloads\full_ba\ba_green_areas_intersect_property.geojson")

In [15]:
data_to_save = overlay_layers(master_data=master_data, root=root, padding_config=padding_config, layers=layers)

roads
buildings
utilities
other_green_areas
pavements
trees_not_over_utilities
trees_over_utilities


In [17]:
data_to_save.shape

(29515, 3)

In [19]:
ruzinov_mask = gpd.read_file(r"C:\Users\mrsam\Downloads\ba2_admin_unit.geojson")

In [21]:
ruzinov_full = clip_polygons(data_to_save, ruzinov_mask)

In [22]:
ruzinov_full.shape

(8673, 21)

In [23]:
data_to_save.to_file('final_ruzinov.geojson', driver='GeoJSON')

In [34]:
layers = [
    ('roads', 'roads'),
    ('buildings', 'buildings'),
    ('utilities', 'combined_utilities_ba'),
    ('other_green_areas', 'ba_green_areas_others'),
    ('pavements', 'ba_pavements'),
    ('trees_not_over_utilities', 'ba_trees_not_over_utilities'),
    ('trees_over_utilities', 'ba_trees_over_utilities'),
]

In [35]:
def clip_polygons(gdf_polygons, gdf_clip_mask):
    """Perform spatial join between polygon data and clipping mask."""
    gdf_polygons = gdf_polygons.to_crs(epsg=4326)
    gdf_intersection = gpd.sjoin(left_df=gdf_polygons, right_df=gdf_clip_mask, how='inner', predicate='within')
    return gdf_intersection

In [36]:
import os
for _, fname in layers:
    print(fname)
    data = gpd.read_file(os.path.join(root, f'{fname}.geojson'))
    print(data.shape)
    if 'index_right' in data.columns:
        data = data.drop(['index_right'], axis=1)
    clipped = clip_polygons(data, ruzinov_mask)
    print(clipped.shape)
    # clipped = gpd.overlay(data, ruzinov_mask, 'intersection')
    clipped.to_file(f'{fname}_clipped2.geojson', driver='GeoJSON')

roads
(41806, 3)
(14649, 20)
buildings
(83898, 3)
(28659, 20)
combined_utilities_ba
(814170, 2)
(235236, 20)
ba_green_areas_others
(21039, 3)
(9439, 20)
ba_pavements
(35557, 228)
(10043, 246)
ba_trees_not_over_utilities
(147591, 2)
(44661, 20)
ba_trees_over_utilities
(6783, 2)
(2758, 20)
